# Import packages

In [ ]:
import numpy as np # numerical computing
import matplotlib.pyplot as plt # plotting
import pandas as pd # data handling
from pathlib import Path # file handling
# the code below autoloads new changes (no restart of the kernel needed)
%reload_ext autoreload
%autoreload 2


# Data loading 

* when you download a dataset, always place it in `../data/` so you can easily open it via relative paths
* try loading it via pandas read options, you can check them out by running `pd.*read*?`
    * e.g. load csv-file `../data/trajectories.csv`: run `df = pd.read_csv("../data/trajectories.csv")`
    * now df is a DataFrame, that contains your data and you can work with it

In [ ]:
# this is the function where we load and already selected the first block of data without any NaN's in it
def load_zebrafish_data(file, only_first_valid_block=True):
    print('I start loading')
    df = pd.read_csv(file, sep='\t')
    df = df.drop(columns=['Unnamed: 24'])
    # now we throw out everything after the first nan occurs
    if only_first_valid_block:
        index_with_first_nan = np.where(df.isna().any(axis=1))[0][0]
        df = df.iloc[:index_with_first_nan]
    print('I am done')
    return df 


In [ ]:
# we load but need to take care of the seperation, since it is a txt document, and not a csv
file_name = '../data/LS1_190_1_2.txt'
df = load_zebrafish_data(file_name, only_first_valid_block=False)

In [ ]:
# subplots is powerful, and sometimes easier to use (here we want to have equal aspect)
f, ax = plt.subplots(1)
ax.plot(df['X1'], df['Y1'], "-o", markersize=1, linewidth=0.5)
ax.set_aspect('equal')
ax.set_axis_off()
f.savefig('zebrafish_avoid_the_center.png')

In [ ]:
# subplots is powerful, or 
f, ax = plt.subplots(1)
ax.plot(df['X5'], df['Y5'], "-o", markersize=1, linewidth=0.5)
ax.set_aspect('equal')

Observation:
* it is important to plot the markers as well (done by "-o" argument in plot)
* by doing so, we see that fish with id 5 jumps from the top to the bottom and back again
* this could be due to an identity switch with another individual

In [ ]:
# now load again but throw everything after the first NaN out
df = load_zebrafish_data(file_name, only_first_valid_block=True)

# Data analysis 
* after the data is cleaned you can start your analysis
* useful commands:
    * `df.mean()`, `df.std()`, `df.min()`, `df.max()`: some basic stats
    * `df.describe()`: the above stats and more in one command
    * `df.plot()`: plot all columns (using index as x-axis)
    * `df.hist()`: 

In [ ]:
# now we create the velocities, for example for individual 1
df['vx1'] = df['X1'].diff()
df['vy1'] = df['Y1'].diff()

In [ ]:
# we use a for loop to do it for all individuals
# we also compute the acceleration
for i in range(1, 8+1):
    df[f'vx{i}'] = df[f'X{i}'].diff()
    df[f'vy{i}'] = df[f'Y{i}'].diff()
    df[f'v{i}'] = np.sqrt( df[f'vx{i}']**2 + df[f'vy{i}']**2 )
    df[f'a{i}'] = df[f'v{i}'].diff()

In [ ]:
# when plotting the first 2 seconds, it becomes clear, that we have too many very burst phases, compare with video
i = 1
f, ax = plt.subplots()
df.head(50)[f'a{i}'].plot(ax=ax)
ax.set(xlabel='time [frames]', ylabel='acceleration [pixel/frame^2]', title='acceleration from raw trajectories (noisy)')
ax.axhline(0)

In [ ]:
# smoothening with a gaussian window
for i in range(1, 8+1):
    df[f'x{i}'] = df[f'X{i}'].rolling(window=5, center=True, min_periods=1, win_type='gaussian').mean(std=1.13)
    df[f'y{i}'] = df[f'Y{i}'].rolling(window=5, center=True, min_periods=1, win_type='gaussian').mean(std=1.13)

In [ ]:
# again comput it via the smoothed coordinates
for i in range(1, 8+1):
    df[f'vx{i}'] = df[f'x{i}'].diff()
    df[f'vy{i}'] = df[f'y{i}'].diff()
    df[f'v{i}'] = np.sqrt(df[f'vx{i}'] ** 2 + df[f'vy{i}'] ** 2)
    df[f'a{i}'] = df[f'v{i}'].diff()


In [ ]:
# plot the first 50 frames (first 4 seconds)
f, ax = plt.subplots()
df.head(50)['a1'].plot(ax=ax)
ax.set(xlabel='time [frames]', ylabel='acceleration [pixel/frame^2]', title='acceleration from smoothed trajectories')
ax.axhline(0, color='k')

## Quantify the drag during coast

In [ ]:
# we only want to plot the drag during coasts (negative accelerations)
coast_accelerations = []
coast_velocities = []
for i in range(1, 8+1):
    mask_negative_acc = df[f'a{i}'] < 0
    coast_accelerations.append(df.loc[mask_negative_acc, f'a{i}'].values)
    coast_velocities.append(df.loc[mask_negative_acc, f'v{i}'].values)


In [ ]:
# 
coast_accelerations = np.concat(coast_accelerations)
coast_velocities = np.concat(coast_velocities)

In [ ]:
f, ax = plt.subplots()
ax.scatter(coast_velocities, coast_accelerations, s=3, alpha=0.2)
ax.set(xlabel='coast_velocities', ylabel='coast_accelerations')
x = np.arange(0, 15)
y = -1/8* x
ax.plot(x, y)

## Nearest neighbor distance

In [ ]:
NNdist = []
for t, row in df.iterrows():
    for i in range(1, 8+1):
        x, y = row[f'x{i}'], row[f'y{i}']
        NNdist_min = np.inf
        for j in range(1, 8+1):
            if i == j:
                continue
            x_other, y_other = row[f'x{j}'], row[f'y{j}']
            dist = np.sqrt( (x-x_other)**2 + (y-y_other)**2 )
            NNdist_min = min(NNdist_min, dist)
        NNdist.append(NNdist_min)


In [ ]:
f, ax = plt.subplots()
_ = ax.hist(NNdist, bins=100)
ax.set(xlabel='Nearest Neighbor distance [pixel]', ylabel='frequency', title='large harvested line')
f.savefig('zebrafish_nn_dist.png')